In [ ]:
def Grid_Search(X_train, y_train,X_test,input_random_seed,currency_name,condition,tf,strategy):

    import numpy as np
    import pandas as pd
    import os
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix
    from IPython.display import clear_output
    from numpy import arange
    import datetime as dt
    from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, train_test_split
    from sklearn.svm import SVC
    from sklearn import model_selection
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import classification_report, confusion_matrix
    
    import pickle
    
    import eli5
    from eli5.sklearn import PermutationImportance
    
    import random
    random.seed(input_random_seed) 
    
    
    param_grid = { 
    'C': [0.1, 1, 10, 100, 1000],  
          'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 
          'kernel': ['rbf']
    }

    
    classifier_SVM = {}
    name = currency_name+"_"+condition+"_"+tf
    
    
    classifier_SVM[name] = SVC()

    
    
    grid_SVM = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3)

    grid_SVM.fit(X_train, y_train)
    best_params = pd.DataFrame()
    best_params = grid_SVM.best_params_
    
    classifier_SVM[name] = SVC(C=best_params['C'],
                               gamma=best_params['gamma'],
                               kernel=best_params['kernel'],probability=True)
    
    model = classifier_SVM[name].fit(X_train,y_train)
    
    
    perm = PermutationImportance(model, random_state=1).fit(X_train,y_train)
    permute_table = eli5.explain_weights_df(perm, feature_names = X_train.columns.tolist())
    
    
    filename = name+'.sav'
    pickle.dump(classifier_SVM[name], open(filename, 'wb'))
    
    
    
    #-----Test-----#
    rfc_predict = classifier_SVM[name].predict_proba(X_test)
    rfc_predict = pd.DataFrame(rfc_predict)


    #----Training-----#
    rfc_predict_train = classifier_SVM[name].predict_proba(X_train)
    rfc_predict_train = pd.DataFrame(rfc_predict_train)

    
    
    
    return rfc_predict,rfc_predict_train,permute_table,best_params
    